In [ ]:
!pip install zarr

In [ ]:
# Import dependencies
from glob import glob
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import zarr

pio.renderers.default = 'iframe'

In [ ]:
# Get experiment runs
runs = sorted(glob('/kaggle/input/czii-cryo-et-object-identification/train/overlay/ExperimentRuns/*'))
runs = [os.path.basename(x) for x in runs]
runs

In [ ]:
# Functions for data loading / visualization

def read_run(run: str) -> pd.DataFrame:
    """Read a experiment run."""
    # Read all types of particle data
    paths = glob(
        f"/kaggle/input/czii-cryo-et-object-identification/train/overlay/ExperimentRuns/{run}/Picks/*.json"
    )
    df = pd.concat([pd.read_json(x) for x in paths]).reset_index(drop=True)

    # Append point information columns
    for axis in "x", "y", "z":
        df[axis] = df.points.apply(lambda x: x["location"][axis])
    for key in "transformation_", "instance_id":
        df[key] = df.points.apply(lambda x: x[key])
    return df

def plot_particles(
    df: pd.DataFrame, scale: float = 1.0, marker_size: float = 2.0
) -> plotly.graph_objs._figure.Figure:
    """Plot 3D scatter plot of particles."""
    df = df.copy()
    df[["x", "y", "z"]] *= scale
    fig = px.scatter_3d(df, x="x", y="y", z="z", color="pickable_object_name")
    fig.update_traces(marker=dict(size=marker_size))
    fig.update_layout(
        title=df.run_name.iloc[0],
        scene=dict(
            yaxis=dict(autorange="reversed"),
            camera=dict(eye=dict(x=1.25, y=-1.25, z=1.25)),
        ),
        width=800,
        height=800,
        template="plotly_dark",
    )
    return fig

def read_zarr(run: str) -> zarr.hierarchy.Group:
    """Read a zarr data (denoised.zarr)."""
    return zarr.open(
        f"/kaggle/input/czii-cryo-et-object-identification/train/static/ExperimentRuns/{run}/VoxelSpacing10.000/denoised.zarr",
        mode="r",
    )

def plot_zarr_images(arr: zarr.core.Array, ncols: int = 6, axsize: float = 2.0):
    """Plot zarr images."""
    nslices = len(arr)
    nrows = math.ceil(nslices / ncols)

    fig = plt.figure(figsize=(axsize * ncols, axsize * nrows))
    for i in range(nslices):
        ax = plt.subplot(nrows, ncols, i + 1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.imshow(arr[i])
    plt.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0.05, hspace=0.05)
    plt.show()


In [ ]:
df = read_run(runs[0])
df

In [ ]:
# Particles
plot_particles(df)

In [ ]:
# Images
zarr_group = read_zarr(runs[0])
plot_zarr_images(zarr_group[2])  # index 2: low resolution array

In [ ]:
# Particles
plot_particles(read_run(runs[1]))

In [ ]:
# Particles
plot_particles(read_run(runs[2]))

In [ ]:
# Images
zarr_group = read_zarr(runs[2])
plot_zarr_images(zarr_group[2])  # index 2: low resolution array

In [ ]:
# Particles
plot_particles(read_run(runs[3]))

In [ ]:
# Images
zarr_group = read_zarr(runs[3])
plot_zarr_images(zarr_group[2])  # index 2: low resolution array

In [ ]:
# Particles
plot_particles(read_run(runs[4]))

In [ ]:
# Images
zarr_group = read_zarr(runs[4])
plot_zarr_images(zarr_group[2])  # index 2: low resolution array

In [ ]:
# Particles
plot_particles(read_run(runs[5]))

In [ ]:
# Images
zarr_group = read_zarr(runs[5])
plot_zarr_images(zarr_group[2])  # index 2: low resolution array

In [ ]:
# Particles
plot_particles(read_run(runs[6]))

In [ ]:
# Images
zarr_group = read_zarr(runs[6])
plot_zarr_images(zarr_group[2])  # index 2: low resolution array

In [ ]:
def _normalize_array(zarr_array: zarr.core.Array) -> np.ndarray:
    """Normalize each slice of the array."""
    arr = np.array(zarr_array)
    mins = arr.min(axis=(1, 2), keepdims=True)
    maxs = arr.max(axis=(1, 2), keepdims=True)
    arr = ((arr - mins) / (maxs - mins) * 255).astype(np.uint8)
    return arr

def _plot_slice_surface(array: zarr.core.Array, z_index: int):
    """Plot surface plot of specified z-slice of array."""
    return go.Surface(
        z=z_index * np.ones((array.shape[1], array.shape[2])),
        surfacecolor=array[z_index],
        colorscale="gray",
        cmin=0,
        cmax=255,
        showscale=False,
    )

def plot_animatable_slices(
    arr: zarr.core.Array,
    step: int = 10,
    init_z: int = 0,
    title: str = "",
    fig: plotly.graph_objs._figure.Figure | None = None,
) -> plotly.graph_objs._figure.Figure:
    """Plot animatable slices."""
    base_traces = list(fig.data) if fig is not None else []

    arr = _normalize_array(arr)
    z_dim = len(arr)

    # Z-indices to annimate
    z_indices = range(0, z_dim, step)

    # Initial plot
    fig = go.Figure(
        data=base_traces + [_plot_slice_surface(arr, init_z)],
        layout=go.Layout(
            title=title,
            scene=dict(
                yaxis=dict(autorange="reversed"),
                zaxis=dict(range=[0, z_dim], autorange=False),
                aspectratio=dict(x=1, y=1, z=1),
                camera=dict(eye=dict(x=1.25, y=-1.25, z=1.25)),
            ),
            width=800,
            height=800,
            template="plotly_dark",
            updatemenus=[
                dict(
                    type="buttons",
                    buttons=[
                        dict(
                            label="▶",
                            method="animate",
                            args=[
                                None,
                                dict(
                                    frame=dict(duration=500, redraw=True),
                                    fromcurrent=True,
                                ),
                            ],
                        )
                    ],
                    font=dict(color="black"),
                )
            ],
        ),
    )

    # Set annimation frames
    frames = []
    for i in z_indices:
        frame = go.Frame(data=base_traces + [_plot_slice_surface(arr, i)], name=str(i))
        frames.append(frame)
    fig.frames = frames

    # Setup slider
    sliders = [
        dict(
            active=0,
            currentvalue=dict(prefix="z-index: "),
            pad=dict(t=50),
            steps=[
                dict(
                    label=str(i),
                    method="animate",
                    args=[
                        [str(i)],
                        dict(
                            mode="immediate",
                            frame=dict(duration=200, redraw=True),
                            transition=dict(duration=0),
                        ),
                    ],
                )
                for i in z_indices
            ],
        )
    ]
    fig.update_layout(sliders=sliders)
    return fig

In [ ]:
run = runs[1]
df = read_run(run)
fig = plot_particles(df, scale=0.1)
zarr_group = read_zarr(run)
plot_animatable_slices(zarr_group[0], title=df.run_name.iloc[0], fig=fig)

In [ ]:
run = runs[2]
df = read_run(run)
fig = plot_particles(df, scale=0.1)
zarr_group = read_zarr(run)
plot_animatable_slices(zarr_group[0], title=df.run_name.iloc[0], fig=fig)

In [ ]:
run = runs[3]
df = read_run(run)
fig = plot_particles(df, scale=0.1)
zarr_group = read_zarr(run)
plot_animatable_slices(zarr_group[0], title=df.run_name.iloc[0], fig=fig)

In [ ]:
run = runs[4]
df = read_run(run)
fig = plot_particles(df, scale=0.1)
zarr_group = read_zarr(run)
plot_animatable_slices(zarr_group[0], title=df.run_name.iloc[0], fig=fig)

In [ ]:
run = runs[5]
df = read_run(run)
fig = plot_particles(df, scale=0.1)
zarr_group = read_zarr(run)
plot_animatable_slices(zarr_group[0], title=df.run_name.iloc[0], fig=fig)

In [ ]:
run = runs[6]
df = read_run(run)
fig = plot_particles(df, scale=0.1)
zarr_group = read_zarr(run)
plot_animatable_slices(zarr_group[0], title=df.run_name.iloc[0], fig=fig)